In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch

# Convert numpy arrays to PyTorch Tensors
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()

print(f"X_train_tensor shape: {X_train_tensor.shape}, dtype: {X_train_tensor.dtype}")
print(f"X_test_tensor shape: {X_test_tensor.shape}, dtype: {X_test_tensor.dtype}")
print(f"y_train_tensor shape: {y_train_tensor.shape}, dtype: {y_train_tensor.dtype}")
print(f"y_test_tensor shape: {y_test_tensor.shape}, dtype: {y_test_tensor.dtype}")

In [ ]:
from torch.utils.data import TensorDataset

# 2. Create TensorDataset objects: Create Datasets: Use TensorDataset to create train_dataset and test_dataset from the tensors.
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

print(f"train_dataset size: {len(train_dataset)}")
print(f"test_dataset size: {len(test_dataset)}")

In [ ]:
from torch.utils.data import DataLoader

# 3. Create DataLoaders: Create DataLoaders: Create train_loader and test_loader with a batch size of 32.
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"train_loader has {len(train_loader)} batches, each of size {batch_size}")
print(f"test_loader has {len(test_loader)} batches, each of size {batch_size}")

In [ ]:
# 4. Print shape of one batch

for images, labels in train_loader:
    print(f"Batch image shape: {images.shape}")
    print(f"Batch label shape: {labels.shape}")
    break

In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

dataiter = iter(train_loader)
images, labels = next(dataiter)

np_images = images.numpy().transpose(0, 2, 3, 1)

fig = plt.figure(figsize=(10, 5))
for i in range(5):
    ax = fig.add_subplot(1, 5, i + 1, xticks=[], yticks=[])
    ax.imshow(np_images[i])
    ax.set_title(f'Age: {int(labels[i].item())}')
plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your model class here: Create a model class with **4 linear layers**
import torch.nn as nn

class AgePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        input_size = 3 * 36 * 36
        hidden_1 = 512
        hidden_2 = 256
        hidden_3 = 128
        output_size = 1

        self.fc1 = nn.Linear(input_size, hidden_1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_1, hidden_2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_2, hidden_3)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(hidden_3, output_size)

    def forward(self, x):
        x = x.view(x.size(0), -1)

        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.relu3(self.fc3(x))
        x = self.fc4(x)
        return x

In [ ]:
# Task 2: Write your training loop here: Create a training loop function
import torch.optim as optim

def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.view(-1, 1)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward() # Backward pass
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(train_loader)

In [ ]:
# Task 3: Write your validation loop here: Create a validation loop function
import torch

def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.view(-1, 1)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)

            running_loss += loss.item()

    return running_loss / len(val_loader)

In [ ]:
# Task 4: Define device, model, loss, optimizer: Define the device, model, loss function, and optimizer
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = AgePredictor().to(device)
print("Model instantiated and moved to device.")

criterion = nn.MSELoss()
print("Loss function (MSELoss) defined.")

learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
print(f"Optimizer (Adam) initialized with learning rate: {learning_rate}")

In [ ]:
# Task 5: Start training for 20 epochs and track training and validation losses
epochs = 20
train_losses = []
val_losses = []

print("Starting training...")

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate_one_epoch(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}')

print("Training complete.")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

epochs_range = range(1, len(train_losses) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs_range, train_losses, label='Training Loss')
plt.plot(epochs_range, val_losses, label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:
import matplotlib.pyplot as plt

dataiter = iter(test_loader)
images, labels = next(dataiter)

images = images.to(device)
model.eval()
with torch.no_grad():
    outputs = model(images)

images_np = images.cpu().numpy().transpose(0, 2, 3, 1)
predicted_ages = outputs.cpu().numpy().flatten()
actual_ages = labels.cpu().numpy()

fig = plt.figure(figsize=(15, 6))
for i in range(5):
    ax = fig.add_subplot(1, 5, i + 1, xticks=[], yticks=[])
    ax.imshow(images_np[i])
    ax.set_title(f'Actual: {int(actual_ages[i])}\nPred: {predicted_ages[i]:.1f}')
plt.tight_layout()
plt.show()